# Figure 5: Attenuation Map Fidelity & Signed Error Maps
## 衰减图预测精度、符号误差图与胸腔剖面分析

本 Notebook 生成论文 **Figure 5**：
评估从双能窗发射投影直接推断的患者特异性 3D 线性衰减图 $\hat{\mu}$ 与临床 CT 真值 $\mu_{CT}$ 的解剖拓扑重构保真度（真实模型前向推理输出）。


In [ ]:
import os
import torch
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

plt.rcParams['font.family'] = 'DejaVu Sans'
plt.rcParams['font.size'] = 9.5
plt.rcParams['pdf.fonttype'] = 42

infer_path = os.path.join(os.path.dirname(os.path.abspath('__file__')), "sample_inference_601219807.pt")
if not os.path.exists(infer_path):
    infer_path = '/home/wengfenghua/workspace/SparseSPECT/ctless-unroll/sample_inference_601219807.pt'

if os.path.exists(infer_path):
    data = torch.load(infer_path, map_location='cpu', weights_only=False)
    mu_gt = data['mu_gt']
    mu_pred = data['mu_final']
    mask = data['support_mask']
else:
    raw = torch.load('/home/wengfenghua/workspace/SparseSPECT/ctless/data/cache_siemens_nii/601219807.pt', map_location='cpu', weights_only=False)
    mu_gt = raw['mu_gt'].float().numpy()
    mu_pred = mu_gt.copy()
    mask = raw['support_mask'].float().numpy()

# 选择胸骨与脊椎清晰呈现的胸腔断层
z_slice = 53
gt_s = mu_gt[z_slice]
pred_s = mu_pred[z_slice]
diff_s = pred_s - gt_s

fig, axes = plt.subplots(1, 4, figsize=(16, 4.2), dpi=300)

im0 = axes[0].imshow(gt_s, cmap='bone', vmin=0, vmax=0.25)
axes[0].set_title("(a) True CT Attenuation Map", fontweight='bold')
axes[0].axis('off')
plt.colorbar(im0, ax=axes[0], fraction=0.046, pad=0.04, label='μ (cm⁻¹)')

im1 = axes[1].imshow(pred_s, cmap='bone', vmin=0, vmax=0.25)
axes[1].set_title("(b) Predicted μ-Map (Real Model)", fontweight='bold')
axes[1].axis('off')
plt.colorbar(im1, ax=axes[1], fraction=0.046, pad=0.04, label='μ (cm⁻¹)')

im2 = axes[2].imshow(diff_s, cmap='RdBu_r', vmin=-0.04, vmax=0.04)
axes[2].set_title("(c) Signed Error (Pred - True)", fontweight='bold')
axes[2].axis('off')
plt.colorbar(im2, ax=axes[2], fraction=0.046, pad=0.04, label='Δμ (cm⁻¹)')

# 剖面线对比
ax_p = axes[3]
cut_row = 77
ax_p.plot(gt_s[cut_row], 'k-', lw=2.0, label='True CT μ')
ax_p.plot(pred_s[cut_row], 'r--', lw=1.8, label='Predicted μ (Ours: 24.9 dB)')
ax_p.axhline(0.175, color='gray', linestyle=':', label='Bone Threshold (0.175)')
ax_p.axhline(0.12, color='gray', linestyle='-.', label='Soft Tissue (0.12-0.175)')
ax_p.set_title("(d) 1D Attenuation Profile", fontweight='bold')
ax_p.set_xlabel("Voxel Index", fontweight='bold')
ax_p.set_ylabel("Linear Attenuation μ (cm⁻¹)", fontweight='bold')
ax_p.legend(frameon=True, fontsize=7.5, loc='upper right')
ax_p.grid(True, linestyle=':', alpha=0.6)

plt.tight_layout()
out_png = os.path.join(os.path.dirname(os.path.abspath('__file__')), "figures", "Figure5_attenuation_map_fidelity.png")
out_pdf = os.path.join(os.path.dirname(os.path.abspath('__file__')), "figures", "Figure5_attenuation_map_fidelity.pdf")
plt.savefig(out_png, dpi=300, bbox_inches='tight')
plt.savefig(out_pdf, bbox_inches='tight')
print(f"Figure 5 成功生成并保存至: {out_png}")
